# 6.1 PyTorch to ONNX — Apply Notebook

## Objective

Master the end-to-end workflow of exporting PyTorch models to ONNX format, validating
correctness, and verifying numerical parity with ONNX Runtime.

**What you will do:**

| # | Exercise | Key Skill |
|---|----------|-----------|
| 1 | Export a simple Linear model | `torch.onnx.export` basics |
| 2 | Export SmallCNN with dynamic batch | Dynamic axes, opset selection |
| 3 | Validate with `onnx.checker` | Model integrity verification |
| 4 | Numerical parity: PyTorch vs ORT | `np.allclose`, tolerance analysis |
| 5 | Inspect the exported graph | Nodes, ops, shapes, initializers |
| 6 | Shape inference on exported model | `onnx.shape_inference` |
| 7 | Performance comparison | Wall-clock latency benchmarking |
| 8 | **Challenge:** multi-input / multi-output export | Advanced export patterns |

### Prerequisites

```
pip install torch onnx onnxruntime numpy
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import os, sys, time, tempfile, warnings
warnings.filterwarnings("ignore")

import numpy as np

import torch
import torch.nn as nn
import torch.onnx

import onnx
from onnx import checker, helper, TensorProto, numpy_helper, shape_inference

import onnxruntime as ort

print(f"PyTorch        : {torch.__version__}")
print(f"ONNX           : {onnx.__version__}")
print(f"ONNX Runtime   : {ort.__version__}")
print(f"NumPy          : {np.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
print("\nSetup complete.")

## Exercise 1 — Export a Simple Linear Model

The simplest possible export: a single fully-connected layer.

A linear model computes:

$$\mathbf{y} = \mathbf{x} W^\top + \mathbf{b}, \quad W \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}},\; \mathbf{b} \in \mathbb{R}^{d_{\text{out}}}$$

`torch.onnx.export` traces the forward pass with a **dummy input** and records
every tensor operation into an ONNX graph.  Key arguments:

| Argument | Purpose |
|----------|---------|
| `export_params` | Embed learned weights inside the `.onnx` file |
| `opset_version` | ONNX operator-set version (higher → more ops) |
| `input_names` / `output_names` | Readable names for graph I/O |
| `dynamic_axes` | Mark dimensions that vary at runtime |

In [ ]:
class SimpleLinear(nn.Module):
    def __init__(self, in_features=16, out_features=4):
        super().__init__()
        self.fc = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.fc(x)


torch.manual_seed(42)
linear_model = SimpleLinear(in_features=16, out_features=4)
linear_model.eval()

dummy = torch.randn(1, 16)

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "simple_linear.onnx")

    torch.onnx.export(
        linear_model,
        dummy,
        path,
        export_params=True,
        opset_version=17,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    )

    m = onnx.load(path)
    checker.check_model(m)
    file_kb = os.path.getsize(path) / 1024

    print(f"Export OK  — {len(m.graph.node)} nodes, {file_kb:.1f} KB")
    print(f"Ops used   : {sorted(set(n.op_type for n in m.graph.node))}")
    print(f"Opset      : {m.opset_import[0].version}")

    # Quick parity
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    x_np = np.random.randn(5, 16).astype(np.float32)
    with torch.no_grad():
        pt_out = linear_model(torch.from_numpy(x_np)).numpy()
    ort_out = sess.run(None, {"input": x_np})[0]

    assert np.allclose(pt_out, ort_out, atol=1e-6), "Parity FAILED"
    print(f"Parity     : max|Δ| = {np.abs(pt_out - ort_out).max():.2e}  ✓")

## Exercise 2 — Export a CNN with Dynamic Batch Dimension

We define the **SmallCNN** used throughout this chapter.  The architecture is:

$$\text{Input}\;[B,3,H,W]
\xrightarrow{\text{Conv}(3{\to}16,k{=}3)}
\xrightarrow{\text{ReLU}}
\xrightarrow{\text{MaxPool}(2)}
\xrightarrow{\text{Conv}(16{\to}32,k{=}3)}
\xrightarrow{\text{ReLU}}
\xrightarrow{\text{AdaptiveAvgPool}(1)}
\xrightarrow{\text{Flatten}}
\xrightarrow{\text{Linear}(32{\to}10)}$$

`AdaptiveAvgPool2d(1)` collapses the spatial dimensions to $1{\times}1$
regardless of input resolution, so the fully-connected head always sees a
32-d vector.

We mark **batch** as dynamic so the same `.onnx` file works with any batch size.

In [ ]:
class SmallCNN(nn.Module):
    """Conv(3→16) → ReLU → MaxPool → Conv(16→32) → ReLU → AdaptiveAvgPool → Flatten → Linear(32→10)"""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.gap   = nn.AdaptiveAvgPool2d(1)
        self.fc    = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = torch.relu(self.conv2(x))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


torch.manual_seed(0)
cnn = SmallCNN(num_classes=10)
cnn.eval()

dummy_img = torch.randn(2, 3, 32, 32)

with tempfile.TemporaryDirectory() as tmp:
    cnn_path = os.path.join(tmp, "small_cnn.onnx")

    torch.onnx.export(
        cnn,
        dummy_img,
        cnn_path,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["image"],
        output_names=["logits"],
        dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    )

    cnn_proto = onnx.load(cnn_path)
    checker.check_model(cnn_proto)

    n_params = sum(p.numel() for p in cnn.parameters())
    print(f"SmallCNN exported  — {n_params:,} parameters")
    print(f"File size          : {os.path.getsize(cnn_path)/1024:.1f} KB")
    print(f"Graph nodes        : {len(cnn_proto.graph.node)}")
    print(f"Opset              : {cnn_proto.opset_import[0].version}")

    # Verify dynamic batch works at runtime
    sess = ort.InferenceSession(cnn_path, providers=["CPUExecutionProvider"])
    for bs in [1, 4, 16]:
        out = sess.run(None, {"image": np.random.randn(bs, 3, 32, 32).astype(np.float32)})[0]
        print(f"  batch={bs:>2}  →  output shape {out.shape}")
        assert out.shape == (bs, 10), f"Unexpected output shape for batch={bs}"

    print("\nDynamic batch verified ✓")

## Exercise 3 — Validate the Exported Model with `onnx.checker`

`onnx.checker.check_model` verifies:

1. **Structural integrity** — all node inputs/outputs are connected.
2. **Type consistency** — tensor element types match operator contracts.
3. **Attribute validity** — required attributes are present and correctly typed.

A model that passes the checker is *syntactically* valid, but may still produce
wrong results (e.g., wrong weights).  Numeric parity is checked separately.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "cnn_check.onnx")
    torch.onnx.export(
        cnn, torch.randn(1, 3, 32, 32), path,
        opset_version=17,
        input_names=["image"], output_names=["logits"],
        dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    )

    model_proto = onnx.load(path)

    # ── 1. Standard checker ─────────────────────────────────────────────
    try:
        checker.check_model(model_proto)
        print("onnx.checker.check_model  : PASSED")
    except onnx.checker.ValidationError as e:
        print(f"Validation error: {e}")

    # ── 2. Check with full_check (stricter) ─────────────────────────────
    try:
        checker.check_model(model_proto, full_check=True)
        print("  full_check=True         : PASSED")
    except onnx.checker.ValidationError as e:
        print(f"  full_check failed: {e}")

    # ── 3. Verify opset is supported ────────────────────────────────────
    opset_v = model_proto.opset_import[0].version
    print(f"  opset_version           : {opset_v}")
    print(f"  IR version              : {model_proto.ir_version}")
    print(f"  producer                : {model_proto.producer_name} v{model_proto.producer_version}")

    # ── 4. Demonstrate a tampered model that FAILS the check ───────────
    bad = onnx.ModelProto()
    bad.CopyFrom(model_proto)
    bad.graph.node[0].input.append("nonexistent_tensor")
    try:
        checker.check_model(bad)
        print("  tampered model          : unexpectedly passed")
    except Exception as e:
        print(f"  tampered model          : correctly rejected ({type(e).__name__})")

    print("\nValidation exercise complete.")

## Exercise 4 — Numerical Parity: PyTorch vs ONNX Runtime

The gold-standard test: run the *same* input through both runtimes and compare.

`np.allclose` uses the combined tolerance:

$$|a_i - b_i| \;\le\; \underbrace{\texttt{atol}}_{\text{absolute}} \;+\; \underbrace{\texttt{rtol}}_{\text{relative}} \cdot |b_i|$$

For float32 exports we typically use $\texttt{atol} = 10^{-5}$, $\texttt{rtol} = 10^{-5}$.

We test across **multiple input distributions** to catch edge cases
(e.g., normalisation near zero).

In [ ]:
def parity_check(model, dummy_shape, name="Model", atol=1e-5, rtol=1e-5):
    """Compare PyTorch and ONNX Runtime outputs across several input regimes."""
    model.eval()
    spatial = dummy_shape[1:]  # everything after batch

    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "parity.onnx")
        torch.onnx.export(
            model, torch.randn(*dummy_shape), path,
            opset_version=17, do_constant_folding=True,
            input_names=["x"], output_names=["y"],
            dynamic_axes={"x": {0: "batch"}, "y": {0: "batch"}},
        )
        sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])

        cases = [
            ("randn  bs=1",   np.random.randn(1,  *spatial).astype(np.float32)),
            ("randn  bs=8",   np.random.randn(8,  *spatial).astype(np.float32)),
            ("randn  bs=32",  np.random.randn(32, *spatial).astype(np.float32)),
            ("zeros  bs=4",   np.zeros((4,  *spatial), dtype=np.float32)),
            ("ones   bs=4",   np.ones((4,   *spatial), dtype=np.float32)),
            ("large  bs=4",   np.random.randn(4, *spatial).astype(np.float32) * 100),
            ("small  bs=4",   np.random.randn(4, *spatial).astype(np.float32) * 1e-3),
        ]

        print(f"{'─'*64}")
        print(f"  Parity report: {name}")
        print(f"{'─'*64}")
        print(f"  {'Case':<18} {'max|Δ|':>10} {'mean|Δ|':>10}  Status")

        all_ok = True
        for label, x_np in cases:
            with torch.no_grad():
                pt = model(torch.from_numpy(x_np)).numpy()
            ort_out = sess.run(None, {"x": x_np})[0]
            d = np.abs(pt - ort_out)
            ok = np.allclose(pt, ort_out, atol=atol, rtol=rtol)
            all_ok &= ok
            print(f"  {label:<18} {d.max():10.2e} {d.mean():10.2e}  {'PASS' if ok else 'FAIL'}")

        assert all_ok, f"{name}: parity check FAILED"
        print(f"  \n  Overall: ALL PASSED")


parity_check(cnn, (1, 3, 32, 32), name="SmallCNN")

## Exercise 5 — Inspect the Exported Graph

An ONNX graph is a DAG of **nodes** (operators) connected by named tensors.

$$\text{Node} = (\texttt{op\_type},\; [\text{inputs}],\; [\text{outputs}],\; \{\text{attributes}\})$$

**Initializers** are weight tensors baked into the protobuf.  Graph inputs that
also appear as initializers are parameters, not user-supplied data.

In [ ]:
from collections import Counter

def inspect_graph(proto, max_nodes=25):
    g = proto.graph
    print(f"Graph name       : {g.name}")
    print(f"IR / opset       : {proto.ir_version} / {proto.opset_import[0].version}")
    print(f"Producer         : {proto.producer_name} {proto.producer_version}")

    # Inputs
    init_names = {i.name for i in g.initializer}
    print(f"\nUser inputs ({len(g.input) - len(init_names)}):")
    for inp in g.input:
        if inp.name in init_names:
            continue
        tt = inp.type.tensor_type
        dims = [d.dim_param or str(d.dim_value) for d in tt.shape.dim]
        print(f"  {inp.name:<20} dtype={TensorProto.DataType.Name(tt.elem_type):<8} shape=[{', '.join(dims)}]")

    print(f"\nOutputs ({len(g.output)}):")
    for out in g.output:
        tt = out.type.tensor_type
        dims = [d.dim_param or str(d.dim_value) for d in tt.shape.dim]
        print(f"  {out.name:<20} dtype={TensorProto.DataType.Name(tt.elem_type):<8} shape=[{', '.join(dims)}]")

    # Initializers
    total_elems = sum(int(np.prod(i.dims)) for i in g.initializer)
    print(f"\nInitializers: {len(g.initializer)} tensors, {total_elems:,} elements ({total_elems*4/1024:.1f} KB fp32)")
    for init in g.initializer[:8]:
        print(f"  {init.name:<30} shape={list(init.dims)}")
    if len(g.initializer) > 8:
        print(f"  ... and {len(g.initializer) - 8} more")

    # Op distribution
    counts = Counter(n.op_type for n in g.node)
    print(f"\nOp distribution ({len(g.node)} nodes):")
    for op, c in counts.most_common():
        print(f"  {op:<20} {c:>3}  {'█' * c}")

    # Node list
    print(f"\nNode list (first {min(max_nodes, len(g.node))}):")
    for i, node in enumerate(g.node[:max_nodes]):
        ins = ', '.join(node.input[:3])
        if len(node.input) > 3:
            ins += f", ...(+{len(node.input)-3})"
        print(f"  [{i:2d}] {node.op_type:<16} ({ins}) → {list(node.output)}")


with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "inspect.onnx")
    torch.onnx.export(
        cnn, torch.randn(1, 3, 32, 32), path,
        opset_version=17, do_constant_folding=True,
        input_names=["image"], output_names=["logits"],
        dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    )
    proto = onnx.load(path)
    inspect_graph(proto)

## Exercise 6 — Shape Inference on the Exported Model

After export, intermediate tensors may have unknown shapes.  
`onnx.shape_inference.infer_shapes` propagates shapes through the graph
so every edge carries full type/shape metadata.

This is critical for:
- Downstream optimizers that fuse ops based on known shapes.
- Debugging mismatches between expected and actual tensor ranks.
- Quantization tools that need shape info to insert Q/DQ nodes.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "shape_infer.onnx")
    torch.onnx.export(
        cnn, torch.randn(1, 3, 32, 32), path,
        opset_version=17, do_constant_folding=True,
        input_names=["image"], output_names=["logits"],
        dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    )
    raw = onnx.load(path)

    # Before shape inference
    n_before = len(raw.graph.value_info)
    print(f"Value-info entries BEFORE shape inference: {n_before}")

    inferred = shape_inference.infer_shapes(raw)

    n_after = len(inferred.graph.value_info)
    print(f"Value-info entries AFTER  shape inference: {n_after}")
    print(f"  → {n_after - n_before} intermediate shapes discovered\n")

    # Display all intermediate shapes
    print("Intermediate tensor shapes:")
    for vi in inferred.graph.value_info:
        tt = vi.type.tensor_type
        dims = []
        for d in tt.shape.dim:
            if d.dim_param:
                dims.append(d.dim_param)
            else:
                dims.append(str(d.dim_value))
        print(f"  {vi.name:<40} [{', '.join(dims)}]")

    # Validate inferred model still passes checker
    checker.check_model(inferred)
    print("\nInferred model passes checker ✓")

    # Save and compare file sizes
    path_inferred = os.path.join(tmp, "shape_inferred.onnx")
    onnx.save(inferred, path_inferred)
    raw_kb = os.path.getsize(path) / 1024
    inf_kb = os.path.getsize(path_inferred) / 1024
    print(f"\nSize before: {raw_kb:.1f} KB | after: {inf_kb:.1f} KB  (Δ {inf_kb - raw_kb:+.1f} KB from shape metadata)")

## Exercise 7 — Performance Comparison: PyTorch vs ORT

We benchmark wall-clock latency for both runtimes.

**Methodology:**
1. Warm-up phase ($W$ iterations) to stabilise caches and JIT.
2. Measurement phase ($N$ iterations), recording each call.
3. Report **median**, **p95**, and **p99** latencies.

$$\text{speedup} = \frac{\text{median}_{\text{PyTorch}}}{\text{median}_{\text{ORT}}}$$

In [ ]:
def benchmark_latency(fn, warmup=50, iters=200):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(iters):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    arr = np.array(times) * 1000  # ms
    return {"median": np.median(arr), "p95": np.percentile(arr, 95),
            "p99": np.percentile(arr, 99), "mean": arr.mean()}


with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "bench.onnx")
    torch.onnx.export(
        cnn, torch.randn(1, 3, 32, 32), path,
        opset_version=17, do_constant_folding=True,
        input_names=["x"], output_names=["y"],
        dynamic_axes={"x": {0: "b"}, "y": {0: "b"}},
    )
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])

    print(f"{'Batch':>6} {'PyTorch med':>14} {'ORT med':>14} {'Speedup':>10}")
    print("─" * 50)

    for bs in [1, 4, 16, 64]:
        x_np = np.random.randn(bs, 3, 32, 32).astype(np.float32)
        x_pt = torch.from_numpy(x_np)

        def run_pt():
            with torch.no_grad():
                cnn(x_pt)

        def run_ort():
            sess.run(None, {"x": x_np})

        pt_stats = benchmark_latency(run_pt)
        ort_stats = benchmark_latency(run_ort)
        speedup = pt_stats["median"] / ort_stats["median"]

        print(f"{bs:>6} {pt_stats['median']:>11.3f} ms {ort_stats['median']:>11.3f} ms {speedup:>9.2f}x")

    print("\n(Speedup > 1 means ORT is faster)")

## Exercise 8 — Challenge: Export a Multi-Input / Multi-Output Model

Real-world models often consume *multiple* inputs (e.g., image + metadata,
query + context) and produce *multiple* outputs (logits, embeddings, attention
weights).

Build a model with:
- **Input 1:** image tensor $[B, 3, 32, 32]$
- **Input 2:** metadata vector $[B, 8]$
- **Output 1:** class logits $[B, 10]$
- **Output 2:** embedding $[B, 64]$

Verify all dynamic axes work and that both outputs match.

In [ ]:
class MultiIOModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.gap   = nn.AdaptiveAvgPool2d(1)
        self.meta_fc = nn.Linear(8, 32)
        self.embed_fc = nn.Linear(64, 64)
        self.cls_fc   = nn.Linear(64, 10)

    def forward(self, image, metadata):
        img_feat = torch.relu(self.conv1(image))
        img_feat = torch.relu(self.conv2(img_feat))
        img_feat = self.gap(img_feat).flatten(1)          # [B, 32]
        meta_feat = torch.relu(self.meta_fc(metadata))    # [B, 32]
        combined = torch.cat([img_feat, meta_feat], dim=1) # [B, 64]
        embedding = torch.relu(self.embed_fc(combined))    # [B, 64]
        logits = self.cls_fc(embedding)                    # [B, 10]
        return logits, embedding


torch.manual_seed(123)
multi_model = MultiIOModel()
multi_model.eval()

dummy_img  = torch.randn(2, 3, 32, 32)
dummy_meta = torch.randn(2, 8)

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "multi_io.onnx")

    torch.onnx.export(
        multi_model,
        (dummy_img, dummy_meta),
        path,
        opset_version=17,
        do_constant_folding=True,
        input_names=["image", "metadata"],
        output_names=["logits", "embedding"],
        dynamic_axes={
            "image":     {0: "batch"},
            "metadata":  {0: "batch"},
            "logits":    {0: "batch"},
            "embedding": {0: "batch"},
        },
    )

    proto = onnx.load(path)
    checker.check_model(proto)
    print(f"Export OK  — inputs: {[i.name for i in proto.graph.input if i.name not in {init.name for init in proto.graph.initializer}]}")
    print(f"             outputs: {[o.name for o in proto.graph.output]}")
    print(f"             nodes: {len(proto.graph.node)}, ops: {sorted(set(n.op_type for n in proto.graph.node))}")

    # Parity check for BOTH outputs
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])

    for bs in [1, 4, 16]:
        img_np  = np.random.randn(bs, 3, 32, 32).astype(np.float32)
        meta_np = np.random.randn(bs, 8).astype(np.float32)

        with torch.no_grad():
            pt_logits, pt_embed = multi_model(
                torch.from_numpy(img_np), torch.from_numpy(meta_np)
            )

        ort_logits, ort_embed = sess.run(None, {"image": img_np, "metadata": meta_np})

        d_logits = np.abs(pt_logits.numpy() - ort_logits).max()
        d_embed  = np.abs(pt_embed.numpy()  - ort_embed).max()

        assert np.allclose(pt_logits.numpy(), ort_logits, atol=1e-5)
        assert np.allclose(pt_embed.numpy(),  ort_embed,  atol=1e-5)

        print(f"  batch={bs:>2}  logits max|Δ|={d_logits:.2e}  embed max|Δ|={d_embed:.2e}  ✓")

    print("\nMulti-input / multi-output parity verified.")

## Summary

| Skill | Key Takeaway |
|-------|--------------|
| Basic export | `torch.onnx.export` traces the forward pass into a static graph |
| Dynamic axes | Mark variable-size dims so one `.onnx` serves all batch sizes |
| Validation | `onnx.checker` catches structural bugs; always run it |
| Parity | Compare across multiple input distributions; use `np.allclose` |
| Graph inspection | Read nodes, ops, shapes, initializers from the protobuf |
| Shape inference | Propagate shapes so optimizers/quantizers can reason about ranks |
| Benchmarking | Warm-up, then measure percentiles for fair latency comparison |
| Multi-IO | Pass tuples for inputs; name every I/O and set dynamic axes for each |